# DETR Training - Industrial Defect Detection
**Ref:** Carion et al., "End-to-End Object Detection with Transformers" (ECCV 2020)

**Classes:** DiVat, DiVatLoiLom, LoiChi, LoiNhua, LoiTray (5 + 1 background)

## Setup
1. Upload dataset (COCO format) to Kaggle as a Dataset.
2. Enable GPU Accelerator (P100 or T4).
3. Run all cells.

In [ ]:
# Cell 1: Imports & Config
import os, json, math, time, glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from scipy.optimize import linear_sum_assignment
from tqdm.notebook import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Cell 2: Hyperparameters
# ========== DATASET ==========
# !!! Thay doi duong dan nay cho phu hop voi Kaggle !!!
TRAIN_ROOT = '/kaggle/input/your-dataset-name/train'
TRAIN_ANN  = '/kaggle/input/your-dataset-name/train/_annotations.coco.json'
VAL_ROOT   = '/kaggle/input/your-dataset-name/valid'
VAL_ANN    = '/kaggle/input/your-dataset-name/valid/_annotations.coco.json'

CLASS_NAMES = ['DiVat', 'DiVatLoiLom', 'LoiChi', 'LoiNhua', 'LoiTray']
NUM_CLASSES = 5

# ========== MODEL ==========
HIDDEN_DIM = 256
NHEAD = 8
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 6
DIM_FEEDFORWARD = 2048
DROPOUT = 0.1
NUM_QUERIES = 100

# ========== TRAINING ==========
BATCH_SIZE = 2
EPOCHS = 50
NUM_WORKERS = 2
LR_TRANSFORMER = 1e-4
LR_BACKBONE = 1e-5
WEIGHT_DECAY = 1e-4
CLIP_MAX_NORM = 0.1
LR_DROP = 40

# ========== LOSS ==========
WEIGHT_CE = 1.0
WEIGHT_BBOX = 5.0
WEIGHT_GIOU = 2.0
EOS_COEF = 0.1
COST_CLASS = 1.0
COST_BBOX = 5.0
COST_GIOU = 2.0

CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Config OK')

In [ ]:
# Cell 3: Backbone - ResNet50
class BackboneResNet50(nn.Module):
    def __init__(self, hidden_dim=256, pretrained=True):
        super().__init__()
        resnet = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        )
        self.backbone = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4,
        )
        self.conv_proj = nn.Conv2d(2048, hidden_dim, kernel_size=1)

    def forward(self, x):
        return self.conv_proj(self.backbone(x))

print('Backbone OK')

In [ ]:
# Cell 4: Positional Encoding (Sine 2D)
class PositionalEncodingSine(nn.Module):
    def __init__(self, hidden_dim=256, temperature=10000.0, normalize=True, scale=2*math.pi):
        super().__init__()
        assert hidden_dim % 2 == 0
        self.hidden_dim = hidden_dim
        self.temperature = temperature
        self.normalize = normalize
        self.scale = scale

    def forward(self, mask):
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        if self.normalize:
            eps = 1e-6
            y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
            x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale
        half_dim = self.hidden_dim // 2
        dim_t = torch.arange(half_dim, dtype=torch.float32, device=mask.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / half_dim)
        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[:,:,:,0::2].sin(), pos_x[:,:,:,1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[:,:,:,0::2].sin(), pos_y[:,:,:,1::2].cos()), dim=4).flatten(3)
        pos = torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)
        return pos

print('PositionalEncoding OK')

In [ ]:
# Cell 5: Transformer Encoder-Decoder
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model=256, nhead=8, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=False)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.activation = nn.ReLU(inplace=True)

    def forward(self, src, pos, src_key_padding_mask=None):
        q = k = src + pos
        attn_output, _ = self.self_attn(q, k, src, key_padding_mask=src_key_padding_mask)
        src = self.norm1(src + self.dropout1(attn_output))
        ffn = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = self.norm2(src + self.dropout2(ffn))
        return src

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model=256, nhead=8, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=False)
        self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=False)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        self.activation = nn.ReLU(inplace=True)

    def forward(self, tgt, memory, pos, query_pos, memory_key_padding_mask=None):
        q = k = tgt + query_pos
        tgt = self.norm1(tgt + self.dropout1(self.self_attn(q, k, tgt)[0]))
        tgt = self.norm2(tgt + self.dropout2(self.multihead_attn(tgt + query_pos, memory + pos, memory, key_padding_mask=memory_key_padding_mask)[0]))
        ffn = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
        tgt = self.norm3(tgt + self.dropout3(ffn))
        return tgt

class DETRTransformer(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.encoder_layers = nn.ModuleList([TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout) for _ in range(num_encoder_layers)])
        self.decoder_layers = nn.ModuleList([TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout) for _ in range(num_decoder_layers)])
        self._reset_parameters()

    def _reset_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, pos, query_embed, mask=None):
        B = src.shape[1]
        memory = src
        for layer in self.encoder_layers:
            memory = layer(memory, pos, src_key_padding_mask=mask)
        N = query_embed.shape[0]
        tgt = torch.zeros(N, B, self.d_model, device=src.device)
        query_pos = query_embed.unsqueeze(1).repeat(1, B, 1)
        for layer in self.decoder_layers:
            tgt = layer(tgt, memory, pos, query_pos, memory_key_padding_mask=mask)
        return tgt

print('Transformer OK')

In [ ]:
# Cell 6: DETR Model
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        dims = [input_dim] + [hidden_dim]*(num_layers-1) + [output_dim]
        self.layers = nn.ModuleList([nn.Linear(dims[i], dims[i+1]) for i in range(num_layers)])
        self.num_layers = num_layers
    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < self.num_layers - 1 else layer(x)
        return x

class DETR(nn.Module):
    def __init__(self, num_classes, hidden_dim=256, nhead=8, num_encoder_layers=6,
                 num_decoder_layers=6, dim_feedforward=2048, dropout=0.1, num_queries=100, pretrained_backbone=True):
        super().__init__()
        self.num_classes = num_classes
        self.num_queries = num_queries
        self.hidden_dim = hidden_dim
        self.backbone = BackboneResNet50(hidden_dim, pretrained_backbone)
        self.pos_encoder = PositionalEncodingSine(hidden_dim)
        self.transformer = DETRTransformer(hidden_dim, nhead, num_encoder_layers, num_decoder_layers, dim_feedforward, dropout)
        self.class_embed = nn.Linear(hidden_dim, num_classes + 1)
        self.bbox_embed = MLP(hidden_dim, hidden_dim, 4, 3)
        self.query_embed = nn.Embedding(num_queries, hidden_dim)

    def forward(self, samples, mask=None):
        B, C, H, W = samples.shape
        features = self.backbone(samples)
        _, _, fH, fW = features.shape
        if mask is None:
            feat_mask = torch.zeros((B, fH, fW), dtype=torch.bool, device=features.device)
        else:
            feat_mask = F.interpolate(mask.unsqueeze(1).float(), size=(fH, fW), mode='nearest').squeeze(1).bool()
        pos = self.pos_encoder(feat_mask)
        src = features.flatten(2).permute(2, 0, 1)
        pos_flat = pos.flatten(2).permute(2, 0, 1)
        mask_flat = feat_mask.flatten(1)
        hs = self.transformer(src, pos_flat, self.query_embed.weight, mask_flat)
        hs = hs.transpose(0, 1)
        return {'pred_logits': self.class_embed(hs), 'pred_boxes': self.bbox_embed(hs).sigmoid()}

print('DETR Model OK')

In [ ]:
# Cell 7: Box Utilities + Loss (SetCriterion)
def box_cxcywh_to_xyxy(boxes):
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx-0.5*w, cy-0.5*h, cx+0.5*w, cy+0.5*h], dim=-1)

def generalized_box_iou(boxes1, boxes2):
    area1 = (boxes1[:,2]-boxes1[:,0]) * (boxes1[:,3]-boxes1[:,1])
    area2 = (boxes2[:,2]-boxes2[:,0]) * (boxes2[:,3]-boxes2[:,1])
    lt = torch.max(boxes1[:, None, :2], boxes2[None, :, :2])
    rb = torch.min(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:,:,0] * wh[:,:,1]
    union = area1[:, None] + area2[None, :] - inter
    iou = inter / (union + 1e-6)
    enclose_lt = torch.min(boxes1[:, None, :2], boxes2[None, :, :2])
    enclose_rb = torch.max(boxes1[:, None, 2:], boxes2[None, :, 2:])
    enclose_wh = (enclose_rb - enclose_lt).clamp(min=0)
    enclose_area = enclose_wh[:,:,0] * enclose_wh[:,:,1]
    return iou - (enclose_area - union) / (enclose_area + 1e-6)

class HungarianMatcher(nn.Module):
    def __init__(self, cost_class=1.0, cost_bbox=5.0, cost_giou=2.0):
        super().__init__()
        self.cost_class = cost_class
        self.cost_bbox = cost_bbox
        self.cost_giou = cost_giou

    @torch.no_grad()
    def forward(self, outputs, targets):
        B, N = outputs['pred_logits'].shape[:2]
        out_prob = outputs['pred_logits'].flatten(0,1).softmax(-1)
        out_bbox = outputs['pred_boxes'].flatten(0,1)
        tgt_ids = torch.cat([t['labels'] for t in targets])
        tgt_bbox = torch.cat([t['boxes'] for t in targets])
        if tgt_ids.shape[0] == 0:
            return [(torch.as_tensor([],dtype=torch.int64), torch.as_tensor([],dtype=torch.int64)) for _ in range(B)]
        cost_class = -out_prob[:, tgt_ids]
        cost_bbox = torch.cdist(out_bbox, tgt_bbox, p=1)
        cost_giou = -generalized_box_iou(box_cxcywh_to_xyxy(out_bbox), box_cxcywh_to_xyxy(tgt_bbox))
        C = (self.cost_class*cost_class + self.cost_bbox*cost_bbox + self.cost_giou*cost_giou).view(B, N, -1).cpu()
        sizes = [len(t['labels']) for t in targets]
        indices = []
        for i, c in enumerate(C.split(sizes, dim=-1)):
            row_ind, col_ind = linear_sum_assignment(c[i].numpy())
            indices.append((torch.as_tensor(row_ind, dtype=torch.int64), torch.as_tensor(col_ind, dtype=torch.int64)))
        return indices

class SetCriterion(nn.Module):
    def __init__(self, num_classes, matcher, weight_ce=1.0, weight_bbox=5.0, weight_giou=2.0, eos_coef=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.matcher = matcher
        self.weight_ce = weight_ce
        self.weight_bbox = weight_bbox
        self.weight_giou = weight_giou
        empty_weight = torch.ones(num_classes + 1)
        empty_weight[-1] = eos_coef
        self.register_buffer('empty_weight', empty_weight)

    def forward(self, outputs, targets):
        indices = self.matcher(outputs, targets)
        # Classification loss
        pred_logits = outputs['pred_logits']
        B, N = pred_logits.shape[:2]
        target_classes = torch.full((B, N), self.num_classes, dtype=torch.int64, device=pred_logits.device)
        for bi, (pi, gi) in enumerate(indices):
            if len(pi) > 0:
                target_classes[bi, pi] = targets[bi]['labels'][gi]
        loss_ce = F.cross_entropy(pred_logits.transpose(1,2), target_classes, weight=self.empty_weight)
        # Box losses
        pred_boxes_list, gt_boxes_list = [], []
        for bi, (pi, gi) in enumerate(indices):
            if len(pi) > 0:
                pred_boxes_list.append(outputs['pred_boxes'][bi, pi])
                gt_boxes_list.append(targets[bi]['boxes'][gi])
        if len(pred_boxes_list) == 0:
            z = torch.tensor(0.0, device=pred_logits.device)
            return {'loss_ce': loss_ce, 'loss_bbox': z, 'loss_giou': z, 'loss_total': self.weight_ce*loss_ce}
        pred_b = torch.cat(pred_boxes_list)
        gt_b = torch.cat(gt_boxes_list)
        loss_bbox = F.l1_loss(pred_b, gt_b, reduction='mean')
        giou = generalized_box_iou(box_cxcywh_to_xyxy(pred_b), box_cxcywh_to_xyxy(gt_b))
        loss_giou = (1 - torch.diag(giou)).mean()
        loss_total = self.weight_ce*loss_ce + self.weight_bbox*loss_bbox + self.weight_giou*loss_giou
        return {'loss_ce': loss_ce, 'loss_bbox': loss_bbox, 'loss_giou': loss_giou, 'loss_total': loss_total}

print('Matcher + Criterion OK')

In [ ]:
# Cell 8: COCO Dataset + Collate
class COCODETRDataset(Dataset):
    def __init__(self, root, ann_file, class_names, augment=True):
        super().__init__()
        self.root = root
        self.augment = augment
        self.class_names = class_names
        with open(ann_file, 'r') as f:
            coco = json.load(f)
        self.cat_id_to_idx = {cat['id']: i for i, cat in enumerate(coco['categories'])}
        self.images = {img['id']: img for img in coco['images']}
        self.img_to_anns = {}
        for ann in coco['annotations']:
            self.img_to_anns.setdefault(ann['image_id'], []).append(ann)
        self.img_ids = sorted(list(self.images.keys()))
        self.mean = [0.485, 0.456, 0.406]
        self.std = [0.229, 0.224, 0.225]
        print(f'[Dataset] {len(self.img_ids)} images, {len(coco["annotations"])} annotations')

    def __len__(self): return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        info = self.images[img_id]
        img = Image.open(os.path.join(self.root, info['file_name'])).convert('RGB')
        orig_w, orig_h = img.size
        anns = self.img_to_anns.get(img_id, [])
        labels, boxes = [], []
        for ann in anns:
            if ann['category_id'] not in self.cat_id_to_idx: continue
            labels.append(self.cat_id_to_idx[ann['category_id']])
            x, y, w, h = ann['bbox']
            boxes.append([(x+w/2)/orig_w, (y+h/2)/orig_h, w/orig_w, h/orig_h])
        labels = torch.tensor(labels, dtype=torch.long)
        boxes = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0,4), dtype=torch.float32)
        if self.augment:
            if torch.rand(1).item() > 0.5:
                img = TF.hflip(img)
                if boxes.shape[0] > 0: boxes[:, 0] = 1.0 - boxes[:, 0]
            short = torch.randint(480, 801, (1,)).item()
        else:
            short = 800
        scale = short / min(orig_w, orig_h)
        if max(orig_w, orig_h) * scale > 1333:
            scale = 1333 / max(orig_w, orig_h)
        img = TF.resize(img, [int(orig_h*scale), int(orig_w*scale)])
        img = TF.normalize(TF.to_tensor(img), self.mean, self.std)
        return img, {'labels': labels, 'boxes': boxes, 'image_id': torch.tensor([img_id])}

def coco_collate_fn(batch):
    images = [b[0] for b in batch]
    targets = [b[1] for b in batch]
    max_h = max(i.shape[1] for i in images)
    max_w = max(i.shape[2] for i in images)
    B = len(images)
    padded = torch.zeros(B, 3, max_h, max_w)
    masks = torch.ones(B, max_h, max_w, dtype=torch.bool)
    for i, img in enumerate(images):
        h, w = img.shape[1], img.shape[2]
        padded[i, :, :h, :w] = img
        masks[i, :h, :w] = False
    return padded, masks, targets

print('Dataset + Collate OK')

In [ ]:
# Cell 9: Build Model, Optimizer, DataLoader
model = DETR(
    num_classes=NUM_CLASSES, hidden_dim=HIDDEN_DIM, nhead=NHEAD,
    num_encoder_layers=NUM_ENCODER_LAYERS, num_decoder_layers=NUM_DECODER_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD, dropout=DROPOUT, num_queries=NUM_QUERIES
).to(device)
print(f'Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

param_dicts = [
    {'params': [p for n, p in model.named_parameters() if 'backbone' not in n and p.requires_grad], 'lr': LR_TRANSFORMER},
    {'params': [p for n, p in model.named_parameters() if 'backbone' in n and p.requires_grad], 'lr': LR_BACKBONE},
]
optimizer = torch.optim.AdamW(param_dicts, weight_decay=WEIGHT_DECAY)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_DROP, gamma=0.1)

matcher = HungarianMatcher(COST_CLASS, COST_BBOX, COST_GIOU)
criterion = SetCriterion(NUM_CLASSES, matcher, WEIGHT_CE, WEIGHT_BBOX, WEIGHT_GIOU, EOS_COEF).to(device)

train_dataset = COCODETRDataset(TRAIN_ROOT, TRAIN_ANN, CLASS_NAMES, augment=True)
val_dataset = COCODETRDataset(VAL_ROOT, VAL_ANN, CLASS_NAMES, augment=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=coco_collate_fn, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=coco_collate_fn, pin_memory=True)

print(f'Train: {len(train_dataset)} images, Val: {len(val_dataset)} images')

In [ ]:
# Cell 10: Training Loop
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_ce': [], 'train_l1': [], 'train_giou': []}

for epoch in range(1, EPOCHS + 1):
    # ===== TRAIN =====
    model.train(); criterion.train()
    train_loss = 0.0; train_ce = 0.0; train_l1 = 0.0; train_giou = 0.0; n_batches = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [Train]')
    for images, masks, targets in pbar:
        images = images.to(device); masks = masks.to(device)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        outputs = model(images, mask=masks)
        loss_dict = criterion(outputs, targets)
        loss = loss_dict['loss_total']
        optimizer.zero_grad(); loss.backward()
        if CLIP_MAX_NORM > 0: nn.utils.clip_grad_norm_(model.parameters(), CLIP_MAX_NORM)
        optimizer.step()
        train_loss += loss.item(); train_ce += loss_dict['loss_ce'].item()
        train_l1 += loss_dict['loss_bbox'].item(); train_giou += loss_dict['loss_giou'].item()
        n_batches += 1
        pbar.set_postfix(loss=f'{loss.item():.4f}', ce=f'{loss_dict["loss_ce"].item():.4f}',
                         l1=f'{loss_dict["loss_bbox"].item():.4f}', giou=f'{loss_dict["loss_giou"].item():.4f}')
    lr_scheduler.step()
    avg_train = train_loss / max(n_batches, 1)
    history['train_loss'].append(avg_train)
    history['train_ce'].append(train_ce / max(n_batches, 1))
    history['train_l1'].append(train_l1 / max(n_batches, 1))
    history['train_giou'].append(train_giou / max(n_batches, 1))

    # ===== VALIDATION =====
    model.eval(); criterion.eval(); val_loss = 0.0; val_n = 0
    with torch.no_grad():
        for images, masks, targets in tqdm(val_loader, desc=f'Epoch {epoch}/{EPOCHS} [Val]'):
            images = images.to(device); masks = masks.to(device)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            outputs = model(images, mask=masks)
            loss_dict = criterion(outputs, targets)
            val_loss += loss_dict['loss_total'].item(); val_n += 1
    avg_val = val_loss / max(val_n, 1)
    history['val_loss'].append(avg_val)

    print(f'Epoch {epoch}: Train={avg_train:.4f} | Val={avg_val:.4f} | LR={optimizer.param_groups[0]["lr"]:.2e}')

    # Save best
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                     'optimizer_state_dict': optimizer.state_dict(), 'loss': best_val_loss,
                     'num_classes': NUM_CLASSES, 'class_names': CLASS_NAMES,
                     'num_queries': NUM_QUERIES, 'hidden_dim': HIDDEN_DIM},
                   os.path.join(CHECKPOINT_DIR, 'best_model.pth'))
        print(f'  >>> Best model saved (val_loss: {best_val_loss:.4f})')

    # Periodic save
    if epoch % 10 == 0:
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                     'optimizer_state_dict': optimizer.state_dict(), 'loss': avg_val},
                   os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch}.pth'))

print('Training Complete!')

In [ ]:
# Cell 11: Plot Training Curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].set_title('Total Loss')

axes[1].plot(history['train_ce'], label='CE Loss')
axes[1].plot(history['train_l1'], label='L1 Loss')
axes[1].plot(history['train_giou'], label='GIoU Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend(); axes[1].set_title('Component Losses')
plt.tight_layout(); plt.show()

In [ ]:
# Cell 12: Inference on Validation Images
from PIL import ImageDraw
COLORS = [(230,25,75),(60,180,75),(255,225,25),(0,130,200),(245,130,48)]

# Load best model
ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'best_model.pth'), map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# Pick 6 random val images
import random
sample_ids = random.sample(range(len(val_dataset)), min(6, len(val_dataset)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, sid in zip(axes.flatten(), sample_ids):
    img_id = val_dataset.img_ids[sid]
    info = val_dataset.images[img_id]
    pil_img = Image.open(os.path.join(VAL_ROOT, info['file_name'])).convert('RGB')
    w, h = pil_img.size
    img_t = TF.normalize(TF.to_tensor(TF.resize(pil_img, 800)), [0.485,0.456,0.406], [0.229,0.224,0.225]).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(img_t)
    probs = out['pred_logits'].softmax(-1)[0, :, :-1]
    keep = probs.max(-1).values > 0.5
    draw = ImageDraw.Draw(pil_img)
    for p, b in zip(probs[keep], out['pred_boxes'][0, keep]):
        cl = p.argmax().item()
        cx, cy, bw, bh = b.cpu()
        x1, y1, x2, y2 = (cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h
        draw.rectangle([x1,y1,x2,y2], outline=COLORS[cl%5], width=3)
        draw.text((x1,y1-12), f'{CLASS_NAMES[cl]} {p[cl]:.2f}', fill=COLORS[cl%5])
    ax.imshow(pil_img); ax.set_title(info['file_name'][:30]); ax.axis('off')
plt.tight_layout(); plt.show()